## Importações

In [2]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from itertools import product

import warnings
warnings.filterwarnings("ignore")

def warn(*args, **kwargs):
    return None

warnings.warn = warn

## Pré-visualização

In [2]:
env = gym.make("FrozenLake-v1", new_step_api=True, render_mode='human', desc=None, map_name="4x4", is_slippery=True)

# Inicie um novo jogo com env.reset()
current_observation = env.reset() # Isto inicia um novo "episódio" e retorna a observação inicia

# A observação atual é apenas a localização atual
print(current_observation) # Observações são apenas um número

0


In [3]:
# o espaço de ação para este ambiente inclui quatro ações distintas

print(f"Número de ações possíveis: {env.action_space}")

print(f"Número de estados: {env.observation_space}")

print(f"Nossa nova ação: {env.action_space.sample()}")

Número de ações possíveis: Discrete(4)
Número de estados: Discrete(16)
Nossa nova ação: 0


In [4]:
# agora agimos! faça isso com a função step

new_action = env.action_space.sample()

observation, reward, done, info, _ = env.step(new_action) # um passo dado pelo agente

# aqui está uma olhada no que recebemos de volta
print(f"Observação: {observation} \nRecompensa: {reward} \nConcluído: {done} \nInformações: {info}")

Observação: 1 
Recompensa: 0.0 
Concluído: False 
Informações: False


In [5]:
# podemos colocar esse processo em um loop for e ver como o jogo progride

current_observation = env.reset() # inicia um novo jogo

num_mov = 5 # número de movimentos

for i in range(num_mov):
  new_action = env.action_space.sample() # igual a uma nova ação
  print(f"Ação {new_action}")
  observation, reward, done, info, _ = env.step(new_action) # percorre a ação e obtém os resultados
  # aqui está uma olhada no que recebemos de volta
  print(f"observação: {observation}, recompensa: {reward}, concluído: {done}, informações: {info}")

Ação 2
observação: 0, recompensa: 0.0, concluído: False, informações: False
Ação 2
observação: 4, recompensa: 0.0, concluído: False, informações: False
Ação 0
observação: 0, recompensa: 0.0, concluído: False, informações: False
Ação 3
observação: 0, recompensa: 0.0, concluído: False, informações: False
Ação 2
observação: 0, recompensa: 0.0, concluído: False, informações: False


In [6]:
# Vamos simular um episódio inteiro

current_observation = env.reset() # inicia um novo jogo
done = False

while not done:
    new_action = env.action_space.sample()
    new_observation, reward, done, info, _ = env.step(new_action)
    print(f"action:{new_action} observation: {new_observation}, reward: {reward}, done: {done}, info: {info}")

action:2 observation: 4, reward: 0.0, done: False, info: False
action:1 observation: 4, reward: 0.0, done: False, info: False
action:2 observation: 8, reward: 0.0, done: False, info: False
action:1 observation: 9, reward: 0.0, done: False, info: False
action:1 observation: 8, reward: 0.0, done: False, info: False
action:0 observation: 8, reward: 0.0, done: False, info: False
action:0 observation: 4, reward: 0.0, done: False, info: False
action:0 observation: 8, reward: 0.0, done: False, info: False
action:2 observation: 9, reward: 0.0, done: False, info: False
action:2 observation: 5, reward: 0.0, done: True, info: False


## Funções

In [104]:
def test_agent(env, agent, goal, epsilon_final=0.01, epsilon_decay=0.01):
    """
    Testa o agente no ambiente até que o total de recompensas atinja o objetivo.

    Parâmetros:
    env: O ambiente do Gym.
    agent: O agente que está sendo testado.
    goal: O objetivo de recompensas totais que deve ser alcançado.
    epsilon_final: O valor final de epsilon para exploração.
    epsilon_decay: A taxa de decaimento de epsilon por episódio.

    Retorna:
    list_rewards: Uma lista com as recompensas acumuladas por episódio.
    """

    total_rewards = 0  # Inicializa o total de recompensas
    list_rewards = []  # Inicializa a lista de recompensas acumuladas
    episodes = 0  # Contador de episódios

    # Jogar episódios até que as recompensas totais alcancem o objetivo
    while total_rewards < goal:
        obs = env.reset()  # Reiniciar o ambiente para um novo episódio
        action = agent.agent_start(obs)  # Iniciar o agente com a observação inicial
        done = False  # Reiniciar a flag de término para cada episódio
        episode_reward = 0  # Reiniciar a recompensa do episódio

        # Loop através de cada passo no episódio
        while not done:
            obs, reward, terminated, _ = env.step(action)  # Executar um passo no ambiente
            episode_reward += reward  # Acumular recompensa do episódio

            if terminated:
                agent.agent_end(reward)  # Encerrar o episódio do agente
                done = True  # Definir a flag de término para sair do loop
            else:
                action = agent.agent_step(reward, obs)  # Continuar o passo do agente

        total_rewards += episode_reward  # Acumular recompensas
        episodes += 1  # Contar os episódios
        list_rewards.append(total_rewards)  # Adicionar recompensas totais à lista

        # Decaimento do epsilon após cada episódio
        if agent.epsilon > epsilon_final:
            agent.epsilon = max(agent.epsilon - epsilon_decay, epsilon_final)

    # Fechar o ambiente
    env.close()

    # Imprimir resultados finais
    print(f"Episódios: {episodes}")
    print(f"Precisão: {episodes/total_rewards}")

    return list_rewards

In [84]:
def run_episodes(env, agent, total_episodes, max_steps):
    rewards = []

    for episode in tqdm(range(total_episodes), desc="Episódios de treinamento"):
        observation = env.reset()
        done = False
        total_reward = 0
        steps = 0

        while not done and steps < max_steps:
            if steps == 0:
                action = agent.agent_start(observation)
            else:
                action = agent.agent_step(reward, observation)

            observation, reward, done, info = env.step(action)
            total_reward += reward
            steps += 1

        agent.agent_end(reward)
        rewards.append(total_reward)

    return rewards

In [45]:
# Função para executar o experimento
def run_experiment(env_name, agents, num_episodes, step_sizes, num_runs, max_steps):
    env = gym.make(env_name)
    all_reward_sums = {}
    cross_product = list(product(agents.keys(), step_sizes, range(num_runs)))

    for agent_name, step_size, run in tqdm(cross_product, desc="Executando experimentos"):
        if (agent_name, step_size) not in all_reward_sums:
            all_reward_sums[(agent_name, step_size)] = []

        agent_class = agents[agent_name]
        agent_info = {
            "env": env,
            "num_actions": env.action_space.n,
            "num_states": env.observation_space.n,
            "epsilon": 0.1,
            "discount": 1.0,
            "step_size": step_size,
            "seed": run
        }
        agent = agent_class(**agent_info)

        total_rewards = []
        for episode in range(num_episodes):
            total_reward = run_episode(env, agent, max_steps)
            total_rewards.append(total_reward)
        all_reward_sums[(agent_name, step_size)].append(np.mean(total_rewards))

    env.close()
    return all_reward_sums, step_sizes

In [47]:
def plot_results(all_reward_sums, step_sizes, algorithms):
    for algorithm in algorithms:
        algorithm_means = np.array([np.mean(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        algorithm_stds = np.array([np.std(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        plt.plot(step_sizes, algorithm_means, marker='o', linestyle='solid', label=algorithm)
        plt.fill_between(step_sizes, algorithm_means + algorithm_stds, algorithm_means - algorithm_stds, alpha=0.2)

    plt.legend()
    plt.xlabel("Step-size")
    plt.ylabel("Sum of\n rewards\n per episode", rotation=0, labelpad=50)
    plt.xticks(step_sizes)
    plt.show()

In [56]:
def plot_rewards_cumulative(rewards):
    cumulative_rewards = [sum(rewards[:i+1]) for i in range(len(rewards))]

    plt.figure(figsize=(12, 6))
    plt.plot(cumulative_rewards)
    plt.xlabel("Episódios")
    plt.ylabel("Recompensa Acumulada")
    plt.title("Recompensa Acumulada ao Longo dos Episódios")
    plt.grid(True)
    plt.show()

## Agentes

In [7]:
class QLearningAgent:
    def __init__(self, env, num_actions, num_states, discount, epsilon, step_size, seed=42, name="QLearningAgent"):
        self.env = env
        self.num_actions = env.action_space.n
        self.num_states = env.observation_space.n
        self.rand_generator = np.random.RandomState(seed)
        self.discount = discount    # Fator de desconto
        self.epsilon = epsilon      # Taxa de exploração
        self.step_size = step_size  # Taxa de aprendizado
        self.seed = seed
        self.name = name
        self.q = np.zeros((self.num_states, self.num_actions)) # Matriz de Q-values

    def agent_start(self, observation):
        state = observation
        current_q = self.q[state, :]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)
        self.prev_state = state
        self.prev_action = action
        return action

    def agent_step(self, reward, observation):
        state = observation
        current_q = self.q[state, :]

        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)

        target = reward + self.discount * np.max(current_q)
        self.q[self.prev_state, self.prev_action] += self.step_size * (target - self.q[self.prev_state, self.prev_action])

        self.prev_state = state
        self.prev_action = action

        return action

    def agent_end(self, reward):
        self.q[self.prev_state, self.prev_action] += self.step_size * (reward - self.q[self.prev_state, self.prev_action])

    def argmax(self, q_values):
        top = float("-inf")
        ties = []

        for i in range(len(q_values)):
            if q_values[i] > top:
                top = q_values[i]
                ties = [i]
            elif q_values[i] == top:
                ties.append(i)

        return self.rand_generator.choice(ties)

In [8]:
class ExpectedSarsaAgent:
    def __init__(self, env, num_actions, num_states, discount, epsilon, step_size, seed=42):
      # Store the parameters provided in agent_init_info.
      self.num_actions = num_actions
      self.num_states = num_states
      self.epsilon = epsilon
      self.step_size = step_size
      self.discount = discount
      self.seed = seed
      self.rand_generator = np.random.RandomState(seed)
      # Create an array for action-value estimates and initialize it to zero.
      self.q = np.zeros((self.num_states, self.num_actions)) # Matriz de Q-value

    def agent_start(self, observation):
        """The first method called when the episode starts, called after
        the environment starts.
        Args:
            observation (int): the state observation from the
                environment's evn_start function.
        Returns:
            action (int): the first action the agent takes.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state, :]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)
        self.prev_state = state
        self.prev_action = action
        return action

    def agent_step(self, reward, observation):
        """A step taken by the agent.
        Args:
            reward (float): the reward received for taking the last action taken
            observation (int): the state observation from the
                environment's step based on where the agent ended up after the
                last step.
        Returns:
            action (int): the action the agent is taking.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state,:]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)

        expected_q = np.sum((self.epsilon / self.num_actions) * current_q)
        best_action = self.argmax(current_q)

        expected_q += (1.0 - self.epsilon) * self.q[state, best_action]

        self.q[self.prev_state, self.prev_action] += self.step_size * (
            reward + self.discount * expected_q - self.q[self.prev_state, self.prev_action])

        self.prev_state = state
        self.prev_action = action
        return action

    def agent_end(self, reward):
        """Run when the agent terminates.
        Args:
            reward (float): the reward the agent received for entering the
                terminal state.
        """
        # Update the Q-value for the last state-action pair.
        self.q[self.prev_state, self.prev_action] += self.step_size * (reward - self.q[self.prev_state, self.prev_action])

    def argmax(self, q_values):
        """argmax with random tie-breaking
        Args:
            q_values (Numpy array): the array of action-values
        Returns:
            action (int): an action with the highest value
        """
        top = float("-inf")
        ties = []

        for i in range(len(q_values)):
            if q_values[i] > top:
                top = q_values[i]
                ties = []

            if q_values[i] == top:
                ties.append(i)

        return self.rand_generator.choice(ties)

## Testes

In [ ]:
np.random.seed(42)

# Parâmetros do experimento
env_name = "FrozenLake-v1"
agents = {
    "Q-learning": QLearningAgent,
    "Expected Sarsa": ExpectedSarsaAgent
}
num_episodes = 1000
step_sizes = np.linspace(0.1, 1.0, 10)
num_runs = 30
max_steps = 99

# Executar o experimento e plotar resultados
all_reward_sums, step_sizes = run_experiment(env_name, agents, num_episodes, step_sizes, num_runs, max_steps)
plot_results(all_reward_sums, step_sizes, list(agents.keys()))

In [ ]:
np.random.seed(42)

env_name = "FrozenLake-v1"
env = gym.make(env_name)

discount = 0.99  # Discounting rate
actions = env.action_space.n
states = env.observation_space.n

total_episodes = 10000
max_steps = 100

epsilons = np.linspace(0.01, 0.1, 10)
step_sizes = np.linspace(0.2, 0.3, 10)

best_mean_reward = -np.inf
best_params = (None, None)
all_results = []

for epsilon in epsilons:
    for step_size in step_sizes:
        agent_info = (env, actions, states, discount, epsilon, step_size)
        agent = ExpectedSarsaAgent(*agent_info)
        rewards = run_episodes(env, agent, total_episodes, max_steps)
        mean_reward = np.mean(rewards)
        all_results.append((epsilon, step_size, mean_reward))
        print(f"Epsilon: {epsilon}, Step_size: {step_size}, Média das Recompensas: {mean_reward}")

        if mean_reward > best_mean_reward:
            best_mean_reward = mean_reward
            best_params = (epsilon, step_size)

        env.close()
        env = gym.make(env_name)  # Reset do ambiente

print(f"\nMelhor combinação de parâmetros: Epsilon: {best_params[0]}, Step_size: {best_params[1]}, Média das Recompensas: {best_mean_reward}")

# Plotar recompensas acumuladas da melhor combinação de parâmetros
agent_info = (env, actions, states, discount, best_params[0], best_params[1])
agent = ExpectedSarsaAgent(*agent_info)
rewards = run_episodes(env, agent, total_episodes, max_steps)
plot_rewards_cumulative(rewards)

In [ ]:
np.random.seed(42)

env = gym.make("FrozenLake-v1")

# Definição dos parâmetros do agente
discount = 0.99    # Discounting rate
epsilon = 0.01     # Exploration rate
step_size = 0.23   # Learning rate

actions = env.action_space.n
states = env.observation_space.n

# Inicialização do agente
agent_info = (env, actions, states, discount, epsilon, step_size)
agent = ExpectedSarsaAgent(*agent_info)

# Executar episódios de treinamento
total_episodes = 10000
max_steps = [100, 150, 200]

for max in max_steps:
  rewards = run_episodes(env, agent, total_episodes, max)
  # Fechar o ambiente
  env.close()
  print(f"Média das Recompensas: {np.mean(rewards)}, max_steps: {max}")
  plot_rewards_cumulative(rewards)

## Avaliação

In [ ]:
# Inicializar o ambiente
env = gym.make("FrozenLake-v1")

np.random.seed(42)

# Definição dos parâmetros do agente
actions = env.action_space.n
states = env.observation_space.n
discount = 0.999      # Discounting rate
epsilon = 1.0         # Exploration rate inicial
step_size = 0.233     # Learning rate

# Inicialização do agente
agent_info = (env, actions, states, discount, epsilon, step_size)
agent = ExpectedSarsaAgent(*agent_info)

# Executar o teste
list_rewards = test_agent(env, agent, goal=5000, epsilon_final=0.01, epsilon_decay=0.01)
plot_rewards_cumulative(list_rewards)